# 🗄️ Project Backend: Microsoft Azure & PostgreSQL Implementation
**Project:** EntiLytics  
**Cloud Provider:** Microsoft Azure for Students

This notebook serves as the formal technical documentation for the EntiLytics database. It covers the initial cloud provisioning via Azure CLI, the relational schema design, and the programmatic connection using SQLAlchemy.

## ☁️ Phase 1: Infrastructure Provisioning
The following steps were performed using the **Azure CLI** to establish the cloud environment. This ensures the backend is hosted on a persistent, managed service rather than a local machine.

### **Deployment Steps:**
1. **Service Registration:** Enabled the `Microsoft.DBforPostgreSQL` namespace for the subscription.
2. **Resource Management:** Created `entilytics-rg` to house all project assets.
3. **Server Deployment:** Provisioned a **Flexible Server** (Burstable B1ms) in the `japanwest` region.
4. **Network Security:** Configured firewall rules (`0.0.0.0`) to allow remote connection from the development environment.

In [ ]:
# The following commands were executed in the terminal to initialize the environment.
# These are kept as a reference for reproducibility.

# Register PostgreSQL Provider
az provider register --namespace Microsoft.DBforPostgreSQL

# Create Resource Group
az group create --name entilytics-rg --location japanwest

# Create the Flexible Server
az postgres flexible-server create \
    --resource-group entilytics-rg \
    --name entilytics-db-server \
    --location japanwest \
    --admin-user enti_admin \
    --admin-password <SECURE_PASSWORD> \
    --sku-name Standard_B1ms \
    --tier Burstable \
    --public-access 0.0.0.0

# Create the logical database
az postgres flexible-server db create \
    --resource-group entilytics-rg \
    --server-name entilytics-db-server \
    --database-name entilytics_db

## 🔐 Phase 2: Python Environment & Security
To maintain security, database credentials are not hardcoded. We utilize a `.env` file and the `python-dotenv` library to inject credentials into the application context.

In [ ]:
# Load .env from the root directory
load_dotenv()

# Database Connection
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:5432/{os.getenv('DB_NAME')}?sslmode=require"

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

## 🐍 SQLAlchemy ORM Models

To prevent ImportError in the main app.py, we define the Python classes that map to the SQL tables above. These allow the application to perform CRUD operations (Create, Read, Update, Delete) using Python objects instead of raw SQL strings.

In [ ]:
def verify_schema():
    schema = """
    # User Management 
    CREATE TABLE account (
        accountid BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        gmail VARCHAR(100) UNIQUE NOT NULL,
        account_role VARCHAR(255) NOT NULL DEFAULT 'user',
        created_at TIMESTAMPTZ DEFAULT NOW()
    );

    # Content Storage 
    CREATE TABLE article (
        articleid BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        title TEXT,
        content TEXT,
        created_at TIMESTAMPTZ DEFAULT NOW()
    );

    # NLP Analysis Results
    CREATE TABLE summary (
        summaryid BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        accountid BIGINT REFERENCES account(accountid),
        articleid BIGINT REFERENCES article(articleid) ON DELETE CASCADE,
        summarytext TEXT
    );

    CREATE TABLE analysis_result (
        resultid BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        articleid BIGINT REFERENCES article(articleid) ON DELETE CASCADE,
        rankings_json TEXT,       -- Stores Ranked Entities & Distances
        entities_all_json TEXT,   -- Stores Raw Extracted Entities
        graph_html TEXT           -- Stores Relationship Network (HTML)
    );

    # User Interactions
    CREATE TABLE annotation (
        annotationid BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        accountid BIGINT REFERENCES account(accountid),
        articleid BIGINT REFERENCES article(articleid) ON DELETE CASCADE,
        note TEXT
    );
    """